# 02 — Feature Engineering: IEEE-CIS Fraud Detection
**Project:** Fraud Detection Pipeline | Week 2

Sections:
1. Load cleaned data
2. Drop high-missing columns
3. Missingness flag features
4. Time features
5. Transaction velocity features
6. Amount aggregation features
7. Interaction features
8. Categorical encoding
9. Handle class imbalance (SMOTE)
10. Time-based train/test split
11. Save processed datasets
12. Feature importance preview

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

FRAUD_COLOR   = '#E24B4A'
LEGIT_COLOR   = '#378ADD'
NEUTRAL_COLOR = '#888780'

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/external', exist_ok=True)
print('Setup complete')

## 1. Load & merge raw data
> Same merge as EDA notebook. We re-run it here so this notebook is fully self-contained.

In [ ]:
DATA_PATH = '../data/raw/'

txn   = pd.read_csv(DATA_PATH + 'train_transaction.csv')
ident = pd.read_csv(DATA_PATH + 'train_identity.csv')
df    = txn.merge(ident, on='TransactionID', how='left')

print(f'Merged shape : {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Fraud rate   : {df.isFraud.mean()*100:.3f}%')
print(f'Memory       : {df.memory_usage(deep=True).sum()/1e6:.0f} MB')

## 2. Drop columns with >80% missing
> From EDA: these columns have almost no information. Keeping them adds noise and memory cost.

In [ ]:
missing_pct  = df.isnull().mean()
drop_cols    = missing_pct[missing_pct > 0.80].index.tolist()
# Never drop the target or ID
drop_cols    = [c for c in drop_cols
                if c not in ('isFraud', 'TransactionID')]

print(f'Columns before : {df.shape[1]}')
print(f'Dropping       : {len(drop_cols)} cols with >80% missing')

df = df.drop(columns=drop_cols)
print(f'Columns after  : {df.shape[1]}')

## 3. Missingness flag features
> Key EDA finding: missing device/identity info is a fraud signal.
> For columns with 10-80% missing, we create a binary flag `col_was_missing` BEFORE imputing.
> Once we impute, the missingness information is lost forever — so we must capture it first.

In [ ]:
missing_pct   = df.isnull().mean()
flag_cols     = missing_pct[
    (missing_pct > 0.10) & (missing_pct < 0.80)
].index.tolist()
flag_cols     = [c for c in flag_cols
                 if c not in ('isFraud', 'TransactionID')]

print(f'Creating missingness flags for {len(flag_cols)} columns...')

for col in flag_cols:
    df[col + '_was_missing'] = df[col].isnull().astype(np.int8)

new_flags = [c for c in df.columns if c.endswith('_was_missing')]
print(f'New flag features created : {len(new_flags)}')
print()
# Verify a flag works as a fraud signal
sample_flag = new_flags[0].replace('_was_missing', '')
mfr = df[df[sample_flag + '_was_missing']==1]['isFraud'].mean()*100
pfr = df[df[sample_flag + '_was_missing']==0]['isFraud'].mean()*100
print(f'Example: {sample_flag}')
print(f'  Fraud rate when missing : {mfr:.2f}%')
print(f'  Fraud rate when present : {pfr:.2f}%')
print(f'  Lift                    : {mfr/(pfr+1e-9):.2f}x')

## 4. Impute remaining missing values
> After flagging, we fill the gaps so models don't crash on NaN.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
# Remove non-feature cols
num_cols = [c for c in num_cols
            if c not in ('isFraud', 'TransactionID', 'TransactionDT')]

print(f'Numeric cols to impute : {len(num_cols)}')
print(f'Categorical cols       : {len(cat_cols)}')

# Median for numeric
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Constant for categorical
for col in cat_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna('missing')

remaining_nulls = df.isnull().sum().sum()
print(f'Remaining nulls        : {remaining_nulls}')
print('Imputation complete')

## 5. Time features
> `TransactionDT` is seconds since Dec 1 2017.
> We extract behavioural time signals — night, weekend, hour.
> EDA showed night-time transactions have elevated fraud rates.

In [ ]:
REF_DATE = pd.Timestamp('2017-12-01')
df['datetime']    = REF_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
df['hour']        = df['datetime'].dt.hour.astype(np.int8)
df['day_of_week'] = df['datetime'].dt.dayofweek.astype(np.int8)
df['day_of_month']= df['datetime'].dt.day.astype(np.int8)
df['month']       = df['datetime'].dt.month.astype(np.int8)
df['is_weekend']  = (df['day_of_week'] >= 5).astype(np.int8)
df['is_night']    = (
    (df['hour'] >= 23) | (df['hour'] <= 5)
).astype(np.int8)
df['is_business_hours'] = (
    (df['hour'] >= 9) & (df['hour'] <= 17) & (df['is_weekend'] == 0)
).astype(np.int8)

# Drop datetime helper — not a model feature
df = df.drop(columns=['datetime'])

time_features = ['hour','day_of_week','day_of_month',
                 'month','is_weekend','is_night','is_business_hours']
print(f'Time features added: {time_features}')

# Quick validation
print()
print('Fraud rate by is_night:')
print(df.groupby('is_night')['isFraud'].mean().mul(100).round(3))

## 6. Transaction velocity features
> How many transactions has this card made recently?
> Fraudsters often make many small test transactions rapidly before a large one.
> This is the most important feature category for fraud detection.
> **Note:** We compute expanding counts as a proxy for rolling windows to avoid future data leakage.

In [ ]:
print('Sorting by transaction time...')
df = df.sort_values('TransactionDT').reset_index(drop=True)

print('Computing velocity features per card (card1)...')
# Cumulative transaction count per card up to current transaction
df['card_txn_count_cumulative'] = (
    df.groupby('card1').cumcount() + 1
).astype(np.int32)

# Cumulative amount per card
df['card_amt_cumulative'] = (
    df.groupby('card1')['TransactionAmt'].cumsum()
).astype(np.float32)

# Time since last transaction for same card
df['time_since_last_txn'] = (
    df.groupby('card1')['TransactionDT'].diff().fillna(0)
).astype(np.float32)

# Flag: is this the first ever transaction for this card?
df['is_first_txn_for_card'] = (
    df['card_txn_count_cumulative'] == 1
).astype(np.int8)

velocity_features = [
    'card_txn_count_cumulative',
    'card_amt_cumulative',
    'time_since_last_txn',
    'is_first_txn_for_card'
]
print(f'Velocity features added: {velocity_features}')

print()
print('Fraud rate by is_first_txn_for_card:')
print(df.groupby('is_first_txn_for_card')['isFraud'].mean().mul(100).round(3))

## 7. Amount aggregation features
> How does this transaction compare to this card's normal behaviour?
> A $5,000 transaction from a card that normally spends $20 is suspicious.
> The z-score captures exactly this — deviation from personal baseline.

In [ ]:
print('Computing per-card amount statistics...')

card_stats = (
    df.groupby('card1')['TransactionAmt']
    .agg(['mean','std','min','max','median'])
    .rename(columns={
        'mean'  : 'card_amt_mean',
        'std'   : 'card_amt_std',
        'min'   : 'card_amt_min',
        'max'   : 'card_amt_max',
        'median': 'card_amt_median',
    })
)

df = df.merge(card_stats, on='card1', how='left')

# Fill std NaN (cards with only 1 transaction)
df['card_amt_std'] = df['card_amt_std'].fillna(0)

# Z-score: how many std deviations from card mean?
df['amount_z_score'] = (
    (df['TransactionAmt'] - df['card_amt_mean'])
    / (df['card_amt_std'] + 1e-6)
).astype(np.float32)

# Ratio: amount vs card mean
df['amount_to_mean_ratio'] = (
    df['TransactionAmt'] / (df['card_amt_mean'] + 1e-6)
).astype(np.float32)

# Log amount — handles heavy right skew
df['log_amount'] = np.log1p(df['TransactionAmt']).astype(np.float32)

agg_features = [
    'card_amt_mean','card_amt_std','card_amt_min',
    'card_amt_max','card_amt_median',
    'amount_z_score','amount_to_mean_ratio','log_amount'
]
print(f'Aggregation features added: {len(agg_features)}')

print()
print('amount_z_score stats by fraud label:')
print(df.groupby('isFraud')['amount_z_score'].describe().round(3))

## 8. Interaction features
> Combine signals that individually are weak but together are strong.
> Example: a large amount at night is more suspicious than either alone.

In [ ]:
# Large amount during night hours
df['large_night_txn'] = (
    (df['is_night'] == 1) &
    (df['TransactionAmt'] > df['card_amt_mean'] * 2)
).astype(np.int8)

# First transaction for card + large amount
df['first_txn_large_amt'] = (
    (df['is_first_txn_for_card'] == 1) &
    (df['TransactionAmt'] > 100)
).astype(np.int8)

# Amount x hour (continuous interaction)
df['amount_x_hour'] = (
    df['log_amount'] * df['hour']
).astype(np.float32)

# High z-score at night
df['high_zscore_night'] = (
    (df['amount_z_score'] > 2) & (df['is_night'] == 1)
).astype(np.int8)

interaction_features = [
    'large_night_txn','first_txn_large_amt',
    'amount_x_hour','high_zscore_night'
]
print(f'Interaction features: {interaction_features}')

print()
print('Fraud rate by large_night_txn:')
print(df.groupby('large_night_txn')['isFraud'].mean().mul(100).round(3))

print()
print('Fraud rate by high_zscore_night:')
print(df.groupby('high_zscore_night')['isFraud'].mean().mul(100).round(3))

## 9. Categorical encoding
> **Target encoding** for high-cardinality columns (email domain, card type).
> Encodes each category as its fraud rate — using 5-fold CV to prevent leakage.
> **Label encoding** for low-cardinality columns (ProductCD, card4, card6).
>
> **Why not one-hot?** With 100+ unique email domains, one-hot creates 100+ sparse columns. Target encoding keeps it to 1 column with real signal.

In [ ]:
def target_encode_cv(df, col, target='isFraud', n_splits=5):
    '''Target encode col using cross-val folds to prevent data leakage.'''
    encoded = pd.Series(index=df.index, dtype=np.float32)
    global_mean = df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=False)

    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        val_fold   = df.iloc[val_idx]
        means = train_fold.groupby(col)[target].mean()
        encoded.iloc[val_idx] = (
            val_fold[col].map(means).fillna(global_mean)
        )
    return encoded


# High-cardinality: target encode
high_card_cols = ['P_emaildomain', 'R_emaildomain', 'card1']
high_card_cols = [c for c in high_card_cols if c in df.columns]

print('Target encoding high-cardinality columns...')
for col in high_card_cols:
    df[col + '_te'] = target_encode_cv(df, col)
    print(f'  {col} -> {col}_te  done')

# Low-cardinality: label encode
low_card_cols = ['ProductCD', 'card4', 'card6']
low_card_cols = [c for c in low_card_cols if c in df.columns]

print()
print('Label encoding low-cardinality columns...')
le = LabelEncoder()
for col in low_card_cols:
    df[col + '_le'] = le.fit_transform(df[col].astype(str))
    print(f'  {col} -> {col}_le  done')

# Drop original string columns
orig_cat = high_card_cols + low_card_cols
df = df.drop(columns=orig_cat, errors='ignore')
print()
print(f'Dropped original string cols: {orig_cat}')

## 10. Feature summary before split

In [ ]:
# Remove non-feature columns
drop_from_features = ['TransactionID', 'TransactionDT', 'isFraud']
feature_cols = [c for c in df.columns
                if c not in drop_from_features]

print(f'Total features built : {len(feature_cols)}')
print()

# Categorise by type
time_feats  = ['hour','day_of_week','day_of_month','month',
               'is_weekend','is_night','is_business_hours']
vel_feats   = ['card_txn_count_cumulative','card_amt_cumulative',
               'time_since_last_txn','is_first_txn_for_card']
amt_feats   = ['card_amt_mean','card_amt_std','card_amt_min',
               'card_amt_max','card_amt_median',
               'amount_z_score','amount_to_mean_ratio','log_amount']
int_feats   = ['large_night_txn','first_txn_large_amt',
               'amount_x_hour','high_zscore_night']
miss_feats  = [c for c in feature_cols if c.endswith('_was_missing')]
enc_feats   = [c for c in feature_cols
               if c.endswith('_te') or c.endswith('_le')]

engineered = (len(time_feats) + len(vel_feats) + len(amt_feats)
              + len(int_feats) + len(miss_feats) + len(enc_feats))
raw_count  = len(feature_cols) - engineered

print(f'  Time features        : {len(time_feats)}')
print(f'  Velocity features    : {len(vel_feats)}')
print(f'  Amount features      : {len(amt_feats)}')
print(f'  Interaction features : {len(int_feats)}')
print(f'  Missingness flags    : {len(miss_feats)}')
print(f'  Encoded categoricals : {len(enc_feats)}')
print(f'  Original raw features: {raw_count}')

## 11. Time-based train/test split
> **Never use random split on fraud data.**
> Fraud patterns evolve — train on the past, test on the future.
> Random split allows future information to leak into training.
> 80% train | 20% test, sorted by TransactionDT.

In [ ]:
df = df.sort_values('TransactionDT').reset_index(drop=True)

split_idx = int(len(df) * 0.80)

X = df[feature_cols]
y = df['isFraud']

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f'Train : {X_train.shape[0]:,} rows  |  '
      f'fraud rate {y_train.mean()*100:.3f}%')
print(f'Test  : {X_test.shape[0]:,} rows  |  '
      f'fraud rate {y_test.mean()*100:.3f}%')

# Verify no data leakage — test transactions must be later than train
train_max_dt = df['TransactionDT'].iloc[:split_idx].max()
test_min_dt  = df['TransactionDT'].iloc[split_idx:].min()
print()
print(f'Latest train TransactionDT  : {train_max_dt:,}')
print(f'Earliest test TransactionDT : {test_min_dt:,}')
print(f'No overlap (leakage check)  : {test_min_dt >= train_max_dt}')

## 12. Handle class imbalance with SMOTE
> **Critical rule:** Apply SMOTE ONLY on the training set, AFTER the split.
> Applying SMOTE before splitting leaks synthetic fraud samples into the test set —
> the most common mistake on fraud projects and an instant red flag in GCC interviews.
>
> SMOTE creates synthetic fraud samples by interpolating between existing fraud rows.

In [ ]:
print(f'Before SMOTE:')
print(f'  Train fraud    : {y_train.sum():,}')
print(f'  Train legit    : {(y_train==0).sum():,}')
print(f'  Fraud rate     : {y_train.mean()*100:.3f}%')

smote = SMOTE(random_state=42, sampling_strategy=0.1)
# sampling_strategy=0.1 means fraud becomes 10% of dataset
# (not 50% - that would be too aggressive for banking models)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print()
print(f'After SMOTE:')
print(f'  Train fraud    : {y_train_sm.sum():,}')
print(f'  Train legit    : {(y_train_sm==0).sum():,}')
print(f'  Fraud rate     : {y_train_sm.mean()*100:.3f}%')
print()
print('Test set unchanged (no SMOTE on test - ever):')
print(f'  Test fraud     : {y_test.sum():,}')
print(f'  Test fraud rate: {y_test.mean()*100:.3f}%')

## 13. Save processed datasets
> Parquet format — 10x smaller than CSV, reads 5x faster, preserves dtypes.

In [ ]:
print('Saving processed datasets...')

X_train_sm_df = pd.DataFrame(X_train_sm, columns=feature_cols)
y_train_sm_s  = pd.Series(y_train_sm, name='isFraud')

X_train_sm_df.to_parquet('../data/processed/X_train.parquet', index=False)
y_train_sm_s.to_frame().to_parquet('../data/processed/y_train.parquet', index=False)
X_test.to_parquet('../data/processed/X_test.parquet',  index=False)
y_test.to_frame().to_parquet('../data/processed/y_test.parquet',  index=False)

# Also save feature column list for API use
import json
with open('../data/processed/feature_cols.json', 'w') as fh:
    json.dump(feature_cols, fh)

print('Saved:')
print('  data/processed/X_train.parquet')
print('  data/processed/y_train.parquet')
print('  data/processed/X_test.parquet')
print('  data/processed/y_test.parquet')
print('  data/processed/feature_cols.json')

import os
for f in ['X_train','y_train','X_test','y_test']:
    size = os.path.getsize(f'../data/processed/{f}.parquet')/1e6
    print(f'  {f}.parquet : {size:.1f} MB')

## 14. Quick feature importance preview
> Train a fast LightGBM on the processed data to see which features matter most.
> This is a sanity check — not our final model (that is notebook 03).

In [ ]:
from lightgbm import LGBMClassifier

print('Training quick LightGBM for feature importance check...')
lgbm_quick = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgbm_quick.fit(X_train_sm_df, y_train_sm_s)

importance_df = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': lgbm_quick.feature_importances_
}).sort_values('importance', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(12, 8))
colors = [
    FRAUD_COLOR if any(tag in f for f in [feat]
                       for tag in ['z_score','velocity','night',
                                   'missing','_te'])
    else NEUTRAL_COLOR
    for feat in importance_df['feature']
]
ax.barh(importance_df['feature'], importance_df['importance'], color=NEUTRAL_COLOR)
ax.set_xlabel('LightGBM Feature Importance (gain)')
ax.set_title('Top 25 features — quick importance check', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/external/fe_feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most important features:')
print(importance_df[['feature','importance']].head(10).to_string(index=False))

## 15. Week 2 summary & next steps

In [ ]:
print('=' * 60)
print('  WEEK 2 — FEATURE ENGINEERING COMPLETE')
print('=' * 60)

print()
print('What we built:')
print(f'  Missingness flags   : {len(miss_feats)} features')
print(f'  Time features       : {len(time_feats)} features')
print(f'  Velocity features   : {len(vel_feats)} features')
print(f'  Amount aggregations : {len(amt_feats)} features')
print(f'  Interaction features: {len(int_feats)} features')
print(f'  Encoded categoricals: {len(enc_feats)} features')
print(f'  Total features      : {len(feature_cols)}')

print()
print('Critical decisions made:')
print('  Time-based split    : no data leakage')
print('  SMOTE after split   : no synthetic leakage into test')
print('  Target encoding CV  : no label leakage in encoding')
print('  Missingness flagged : before imputation, not after')

print()
print('Saved to data/processed/ — ready for notebook 03')
print()
print('NEXT -> 03_modelling.ipynb')
print('  LightGBM + XGBoost + Optuna tuning + MLflow tracking')
print('=' * 60)